# ShopDesk, Module 3 Section 4 Lab 2: Guided Troubleshooting

The capstone's second half: a broken ShopDesk setup, fixed step by step. You diagnose why a rule is not
loading (a bad glob), why the wrong tool gets picked (overlapping descriptions), and why an MCP server will
not connect (an unexpanded config value), then fix each and validate. Every diagnosis and fix runs offline
and deterministically, mirroring `/memory`, tool-selection, and `.mcp.json` checks. A live cell confirms the
repaired setup. Runs **Sonnet** (`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

A teammate reports three ShopDesk problems: the testing rule never applies, a refund question keeps calling
the order tool, and the payments MCP server fails to connect. Each has a specific, findable cause. Good
troubleshooting is not guessing; it is inspecting what actually loaded, measuring what actually overlaps, and
validating what actually expands.

The question this lab answers: **how do you diagnose and fix the three most common Claude Code setup
failures?**

## Objectives

- Use a `/memory`-style inspector to find a **rule that is not loading** and fix its glob.
- Diagnose a **misrouted tool** from overlapping descriptions and sharpen it.
- Resolve an **MCP connection error** from an unexpanded config value, and categorize it correctly.

## What you'll observe

- The testing rule is missing for a test file until the glob is corrected.
- Two tools overlap heavily and the wrong one is picked, until the description is sharpened.
- The MCP config leaves a value unexpanded (a config error, not a transient one) until the variable is set.

## How to run

Run top to bottom. Every diagnosis and fix runs anywhere. The final live cell confirms the repaired setup,
so paste a real key into **Setup 2/3** and re-run from the top; otherwise it skips.

## 0. Setup

**This cell:** installs the packages. `pyyaml` parses rule frontmatter; the Agent SDK drives the final
confirmation cell and needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the packages =====
%pip install -q claude-agent-sdk anthropic python-dotenv pyyaml

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` for the live confirmation.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # filesystem paths for the broken sandbox
import re                                       # glob matching and variable expansion
import sys                                       # detect Windows (special event loop)
import yaml                                     # parse rule frontmatter
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cell will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds the **broken** sandbox: a testing rule whose glob has a typo, and an `.mcp.json`
whose server URL references a variable that is not set. The tool problem is set up in code further down. These
are the three faults you will fix.

In [ ]:
# ===== SETUP 3/3 - create the broken setup =====
import textwrap                                    # keeps the embedded file bodies readable
PROJECT = os.path.join(os.getcwd(), "shopdesk_broken")   # the broken project root

def write(rel, content):                           # small helper: write a file under PROJECT
    path = os.path.join(PROJECT, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)

write(".claude/rules/testing.md", textwrap.dedent("""\
    ---
    paths:
      - "**/*.tset.tsx"
    ---
    # Testing rules
    - Mock external services.
    """))                                           # BUG: "tset" should be "test"
write("src/components/Button.test.tsx", "// a test file\n")
write(".mcp.json", json.dumps({
    "mcpServers": {"payments": {"type": "http", "url": "${SHOPDESK_MCP_URL}"}}
}, indent=2))                                       # BUG: the variable is not set anywhere
print("broken sandbox at", PROJECT)

### Troubleshooting techniques

- **Configuration precedence:** use `/memory` to see what actually loaded; a rule that seems missing usually
  has a glob that does not match.
- **Tool selection:** overlapping or vague descriptions make Claude pick the wrong tool; measure the overlap
  and sharpen the description.
- **MCP failures:** a server that will not connect often has an unexpanded `${VAR}`; that is a config error to
  fix, not a transient one to retry.
- Also watch for **context overflow** (trim CLAUDE.md, use path-scoped rules and subagents) and
  **non-determinism** (pin the model, fix ambiguous descriptions).

---

### Lab objective - diagnose, fix, validate

**What you build:** three diagnose-fix-validate loops over the broken setup.

**Why it helps you build real solutions:** these three faults account for most "why is Claude ignoring my
config" reports, and each has a precise fix.

**How you'll see it:** each check flips from failing to passing after the fix.

**This cell:** **Problem 1, a rule that will not load.** We match the test file against the rule's glob.
The typo'd pattern (`tset`) does not match, so a `/memory`-style view shows the testing rule missing, which
is exactly the symptom your teammate saw.

In [ ]:
# ===== diagnose: the testing rule does not load =====
def glob_to_regex(pat):                            # glob -> regex (** spans dirs, * within a segment)
    out = ""; i = 0
    while i < len(pat):
        if pat[i:i+3] == "**/": out += "(?:.*/)?"; i += 3
        elif pat[i:i+2] == "**": out += ".*"; i += 2
        elif pat[i] == "*": out += "[^/]*"; i += 1
        elif pat[i] == "?": out += "[^/]"; i += 1
        else: out += re.escape(pat[i]); i += 1
    return re.compile("^" + out + "$")

def rule_paths(rel):                               # read the paths list from a rule's frontmatter
    text = open(os.path.join(PROJECT, rel)).read()
    m = re.match(r"^---\n(.*?)\n---\n", text, re.DOTALL)
    return (yaml.safe_load(m.group(1)) or {}).get("paths", []) if m else []

TEST_FILE = "src/components/Button.test.tsx"
paths = rule_paths(".claude/rules/testing.md")
loads = any(glob_to_regex(p).match(TEST_FILE) for p in paths)
print("testing.md paths:", paths)
print("loads for the test file?", loads, "(symptom: rule missing in /memory)")

**This cell:** **fix and validate Problem 1.** We correct the glob to `**/*.test.tsx` and re-check. The
rule now matches the test file, so it would load, and `/memory` would show it.

In [ ]:
# ===== fix: correct the glob, then re-check =====
write(".claude/rules/testing.md", textwrap.dedent("""\
    ---
    paths:
      - "**/*.test.tsx"
    ---
    # Testing rules
    - Mock external services.
    """))                                           # fixed: "test" not "tset"
paths = rule_paths(".claude/rules/testing.md")     # re-read
loads = any(glob_to_regex(p).match(TEST_FILE) for p in paths)
print("testing.md paths:", paths)
print("loads for the test file now?", loads)

**This cell:** **Problem 2, a misrouted tool.** Two tools have nearly identical descriptions, so a refund
question can match the order tool. We measure the word overlap and simulate the selection to reproduce the
misroute.

In [ ]:
# ===== diagnose: overlapping descriptions misroute the tool =====
TOOLS = {
    "get_order":  "Look up order information and details for a customer.",
    "get_refund": "Look up order information and details for a customer.",   # BUG: same as get_order
}
def words(s):  return set(re.findall(r"[a-z]+", s.lower()))
def overlap(a, b):                                 # Jaccard overlap of two descriptions
    wa, wb = words(a), words(b)
    return len(wa & wb) / len(wa | wb)

def pick_tool(query):                              # pick the tool whose description shares most words
    qw = words(query)
    return max(TOOLS, key=lambda t: len(qw & words(TOOLS[t])))

print("description overlap:", round(overlap(TOOLS["get_order"], TOOLS["get_refund"]), 2))
print("query 'process a refund' -> picked:", pick_tool("process a refund for this customer"))

**This cell:** **fix and validate Problem 2.** We rewrite the refund tool's description to be specific
and non-overlapping. The overlap drops and the refund query now routes to `get_refund`.

In [ ]:
# ===== fix: sharpen the refund tool description, then re-check =====
TOOLS["get_refund"] = "Process or check a refund and its 30-day eligibility window for an order."   # specific now
print("description overlap:", round(overlap(TOOLS["get_order"], TOOLS["get_refund"]), 2))
print("query 'process a refund' -> picked:", pick_tool("process a refund for this customer"))

**This cell:** **Problem 3, an MCP connection error.** The server URL is `${SHOPDESK_MCP_URL}`, but the
variable is not set, so it stays unexpanded and the connection fails. We expand the config and report any
missing variables, then categorize the failure.

In [ ]:
# ===== diagnose: the MCP server config will not expand =====
VAR_RE = re.compile(r"\$\{([A-Z_][A-Z0-9_]*)(:-(.*?))?\}")   # ${VAR} or ${VAR:-default}

def expand(value, env):                            # expand ${VAR}; collect any missing names
    missing = []
    def repl(m):
        name, _, default = m.group(1), m.group(2), m.group(3)
        if name in env:  return env[name]
        if default is not None: return default
        missing.append(name); return m.group(0)     # leave it literal and flag it
    return VAR_RE.sub(repl, value), missing

cfg = json.load(open(os.path.join(PROJECT, ".mcp.json")))       # read the config
url = cfg["mcpServers"]["payments"]["url"]                       # the templated URL
expanded, missing = expand(url, dict(os.environ))               # try to expand it
category = "config (fix the value); not transient (do not retry)" if missing else "ok"
print("url:", url, "-> expanded:", expanded)
print("missing variables:", missing, "| error category:", category)

**This cell:** **fix and validate Problem 3.** We set the variable (in real life, in your environment or
CI secrets), re-expand, and confirm the URL resolves with nothing missing.

In [ ]:
# ===== fix: set the variable, then re-expand =====
env = dict(os.environ, SHOPDESK_MCP_URL="https://mcp.shopdesk.example/payments")   # provide the value
expanded, missing = expand(url, env)               # expand again with the variable set
print("expanded:", expanded)
print("missing variables:", missing or "none", "| connects now?", not missing)

**This cell:** **validate the whole setup.** We re-run all three checks together and confirm every one
passes, the sign that the repaired configuration is healthy.

In [ ]:
# ===== validate: all three green =====
rule_ok = any(glob_to_regex(p).match(TEST_FILE) for p in rule_paths(".claude/rules/testing.md"))
tool_ok = pick_tool("process a refund for this customer") == "get_refund"
mcp_ok = not expand(url, env)[1]
print("rule loads:", rule_ok)
print("tool routes correctly:", tool_ok)
print("mcp config expands:", mcp_ok)
print("all green:", rule_ok and tool_ok and mcp_ok)

**This cell:** a live confirmation. We point the Agent SDK at the repaired project with
`setting_sources=["project"]` and ask about the testing conventions for the test file, confirming the
now-fixed rule is in scope.

In [ ]:
# ===== live: confirm the repaired setup =====
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock

FIX_OPTS = ClaudeAgentOptions(                     # load the repaired project rules
    model=MODEL, cwd=PROJECT,
    setting_sources=["project"], allowed_tools=["Read", "Grep", "Glob"])

async def ask(prompt):                              # stream just the text answer
    async for m in query(prompt=prompt, options=FIX_OPTS):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:160])

if RUN_LIVE:                                        # needs a real key (and Node.js 18+)
    run_async(lambda: ask("Open src/components/Button.test.tsx and state the testing rule that applies."))
else:
    print("[skipped] expected: the corrected testing rule is now in scope for the .test.tsx file.")

**In the real Claude Code CLI** (reference, not run here):

```text
/memory                 # inspect what rules and memory actually loaded
claude mcp list         # list configured MCP servers and their status
claude mcp get payments # show one server's resolved config
```

Diagnose with these before changing anything; most "Claude ignores my config" reports are a non-matching
glob, an overlapping tool description, or an unexpanded variable.

| symptom | likely cause | fix |
|---|---|---|
| a rule never applies | glob does not match the file | correct the `paths` pattern |
| the wrong tool is called | overlapping or vague descriptions | make each description specific |
| an MCP server will not connect | unexpanded `${VAR}` | set the variable; it is a config error |
| Claude ignores instructions | context overflow | trim CLAUDE.md; use path-scoped rules |

**Lesson:** troubleshooting is inspection, not guessing. Check what loaded with `/memory`, measure tool
overlap when routing goes wrong, and expand `.mcp.json` to catch a missing variable. Each of the three common
failures has a precise cause and a precise fix, and validation is just re-running the check.

---

## Recap - the troubleshooting loop

| Step | Tool | What it tells you |
|---|---|---|
| inspect | `/memory` and glob check | which rules actually load |
| measure | description overlap | why a tool is misrouted |
| expand | `.mcp.json` variable expansion | why a server will not connect |
| validate | re-run the checks | the fix holds |

One principle to carry forward: **diagnose from what actually loaded, fix the specific cause, then re-run the
check to confirm.** To run live, paste a real key into **Setup 2/3** and re-run from the top. That completes
the course: you can now choose the right mechanism, assemble an architecture, and repair a broken setup.